# 2. Autoría visual, Python y SQL → una DSL

La biblioteca no ejecuta el código de autoría. Analiza AST, resuelve referencias contra tipos declarados y rechaza construcciones fuera del subconjunto. El editor visual futuro utilizará estos mismos constructores y validadores.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Ejecutar desde el proyecto o notebooks/"
import json
from dataclasses import asdict
from rule_manager.examples import input_definition, evaluation_definition, PYTHON_RULE, SQL_RULE
from rule_manager.authoring import compile_python, compile_sql
from rule_manager.validation import validate_evaluations
from rule_manager.errors import CompileError
inputs = input_definition()
package = evaluation_definition(inputs)
_, env = validate_evaluations(package, inputs)
visual = package["rules"][0]["expression"]
python_dsl = compile_python(PYTHON_RULE, env)
sql_dsl = compile_sql(SQL_RULE, env)
assert visual == python_dsl == sql_dsl
print(PYTHON_RULE)
print(SQL_RULE)
print(json.dumps(visual, indent=2))

from rule_manager.api import compile_rule
artifact = compile_rule(PYTHON_RULE, "python", env)
assert artifact.expression == visual
print("API compartida: hash de fuente", artifact.source_hash)


def regla(inputs, variables, constants) -> bool:
    return inputs["age"] >= constants["minimum_age"]

inputs.age >= constants.minimum_age
{
  "op": "gte",
  "args": [
    {
      "ref": {
        "scope": "input",
        "path": [
          "age"
        ]
      }
    },
    {
      "ref": {
        "scope": "constant",
        "path": [
          "minimum_age"
        ]
      }
    }
  ]
}
API compartida: hash de fuente 14bcc31d557591757ab7dd21f5304eacf5c8a066f9e53c8745f116975b89359f


## Control de flujo y operaciones sobre arrays

Solo se permiten llamadas propias del lenguaje, sin imports. `exists` acepta una lambda restringida; SQL usa `DSL_EXISTS`. Las expresiones se compilan al mismo árbol.

In [2]:
py_source = 'def regla(inputs) -> bool:\n    return exists(inputs["accounts"], lambda item: item["balance"] > 0)'
sql_source = "DSL_EXISTS(inputs.accounts, item.balance > 0)"
assert compile_python(py_source, env) == compile_sql(sql_source, env)
print(json.dumps(compile_python(py_source, env), indent=2))


{
  "op": "exists",
  "args": [
    {
      "ref": {
        "scope": "input",
        "path": [
          "accounts"
        ]
      }
    },
    {
      "op": "gt",
      "args": [
        {
          "ref": {
            "scope": "item",
            "path": [
              "balance"
            ]
          }
        },
        {
          "literal": {
            "value": 0,
            "data_type": {
              "type": "integer",
              "nullable": false
            }
          }
        }
      ]
    }
  ]
}


In [3]:
bad_sources = {
    "retorno numérico": "def regla(inputs) -> bool:\n    return 1",
    "importación": "import os\ndef regla(inputs) -> bool:\n    return True",
    "campo inexistente": 'def regla(inputs) -> bool:\n    return inputs["unknown"] > 0',
    "camino sin retorno": 'def regla(inputs) -> bool:\n    if inputs["age"] > 18:\n        return True',
    "comparación nullable": 'def regla(inputs) -> bool:\n    return inputs["financial"]["income"] > 0',
}
for name, source in bad_sources.items():
    try:
        compile_python(source, env)
        raise AssertionError("Se aceptó una regla inválida")
    except CompileError as error:
        print(name, asdict(error.diagnostic))


retorno numérico {'code': 'NON_BOOLEAN_RETURN', 'message': 'Every return path must be non-null Boolean', 'path': '$', 'line': 1, 'column': 1}
importación {'code': 'UNSUPPORTED_SYNTAX', 'message': 'Exactly one function; no imports or module-level statements', 'path': '$', 'line': None, 'column': 1}
campo inexistente {'code': 'UNKNOWN_FIELD', 'message': 'Unknown field path: unknown', 'path': '$', 'line': 1, 'column': 1}
camino sin retorno {'code': 'NON_BOOLEAN_RETURN', 'message': 'A path implicitly returns None', 'path': '$', 'line': 1, 'column': None}
comparación nullable {'code': 'NON_BOOLEAN_RETURN', 'message': 'Every return path must be non-null Boolean', 'path': '$', 'line': 1, 'column': 1}


## Documentos JSON exportables

El input y el paquete de evaluación fijan versiones y esquema. Son archivos completos validados, reutilizables por la futura UI. Escribir un archivo aquí no representa aprobación ni publicación en el banco.

In [4]:
destination = ROOT / "examples"
destination.mkdir(exist_ok=True)
for filename, definition in [("inputs.json", inputs), ("evaluations.json", package)]:
    (destination / filename).write_text(json.dumps(definition, indent=2, ensure_ascii=False) + "\n")
print("Ejemplos guardados en", destination)
print("Variables:", [v["id"] for v in package["variables"]])
print("Productos:", [p["id"] for p in package["products"]])


Ejemplos guardados en /Users/manuelrodval/projects/tinker/inhouse-rule-manager/examples
Variables: ['income', 'debt', 'debt_ratio', 'total_balance', 'account_count']
Productos: ['basic_account', 'demo_credit', 'savings_offer']
